# Add Lateral Walls to Equilibrated Gel Slab

Adds four vertical (z-normal) lateral walls (type 6) to an equilibrated LAMMPS gel slab data file. Preserves all angles if present in the input file.

**Atom types:** 1=polymer, 2=crosslinker, 3=solvent, 4=support, 5=piston, 6=walls

In [1]:
import numpy as np
from scipy.spatial import ConvexHull, KDTree

## Parameters

In [2]:
input_file  = "../../lammps_data_files_local/final_config_slab_support_tall_angle_8_1.0_1.0_10000000.data"
output_file = "../../lammps_data_files_local/walled_slab_support_tall_angle_8_1.0_1.0_10000000.data"
wall_spacing = 0.2

## Group / Type Definitions

In [3]:
POLYMER_TYPES = {1, 2}       # group polymer
SOLVENT_TYPES = {3}          # group solvent
SUPPORT_TYPES = {4}          # group support
PISTON_TYPES  = {5}          # group piston
WALL_TYPE     = 6            # group walls

FROZEN_TYPES   = SUPPORT_TYPES | PISTON_TYPES  # existing rigid bodies
ROTATE_TYPES   = POLYMER_TYPES | SOLVENT_TYPES  # group mobile
WALL_CLEARANCE = 0.2                             # gap between gel and walls (σ)
OVERLAP_CUTOFF = 0.8                             # remove solvent within this distance of wall/support/piston (σ)


## LAMMPS I/O

In [4]:
def parse_lammps_data(filename):
    """
    Parse a LAMMPS data file (molecular style).
    Returns atoms, bonds, angles, box_bounds, masses, header_info.
    angles is an empty list if no Angles section is present.
    """
    atoms   = []
    bonds   = []
    angles  = []
    box_bounds  = {}
    masses      = {}
    header_info = {}

    with open(filename, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # --- Header counts ---
        if line.endswith('atoms') and len(line.split()) == 2:
            header_info['natoms'] = int(line.split()[0])
        elif line.endswith('bonds') and len(line.split()) == 2:
            header_info['nbonds'] = int(line.split()[0])
        elif line.endswith('angles') and len(line.split()) == 2:
            header_info['nangles'] = int(line.split()[0])
        elif 'atom types' in line:
            header_info['atom_types'] = int(line.split()[0])
        elif 'bond types' in line:
            header_info['bond_types'] = int(line.split()[0])
        elif 'angle types' in line:
            header_info['angle_types'] = int(line.split()[0])

        # --- Box bounds ---
        elif 'xlo xhi' in line:
            parts = line.split()
            box_bounds['xlo'] = float(parts[0]); box_bounds['xhi'] = float(parts[1])
        elif 'ylo yhi' in line:
            parts = line.split()
            box_bounds['ylo'] = float(parts[0]); box_bounds['yhi'] = float(parts[1])
        elif 'zlo zhi' in line:
            parts = line.split()
            box_bounds['zlo'] = float(parts[0]); box_bounds['zhi'] = float(parts[1])

        # --- Masses ---
        elif line == 'Masses':
            i += 2
            while i < len(lines) and lines[i].strip() and \
                  not lines[i].strip().split()[0].isalpha():
                parts = lines[i].split()
                if len(parts) >= 2:
                    try:
                        masses[int(parts[0])] = float(parts[1])
                    except ValueError:
                        pass
                i += 1
            continue

        # --- Atoms ---
        elif line == 'Atoms' or line.startswith('Atoms '):
            i += 2
            while i < len(lines) and lines[i].strip() and \
                  not lines[i].strip().split()[0].isalpha():
                parts = lines[i].split()
                if len(parts) >= 6:
                    try:
                        atoms.append({
                            'id':  int(parts[0]),
                            'mol': int(parts[1]),
                            'type': int(parts[2]),
                            'x': float(parts[3]),
                            'y': float(parts[4]),
                            'z': float(parts[5])
                        })
                    except ValueError:
                        pass
                i += 1
            continue

        # --- Bonds ---
        elif line == 'Bonds' or line.startswith('Bonds '):
            i += 2
            while i < len(lines) and lines[i].strip() and \
                  not lines[i].strip().split()[0].isalpha():
                parts = lines[i].split()
                if len(parts) >= 4:
                    try:
                        bonds.append({
                            'id':    int(parts[0]),
                            'type':  int(parts[1]),
                            'atom1': int(parts[2]),
                            'atom2': int(parts[3])
                        })
                    except ValueError:
                        pass
                i += 1
            continue

        # --- Angles ---
        elif line == 'Angles' or line.startswith('Angles '):
            i += 2
            while i < len(lines) and lines[i].strip() and \
                  not lines[i].strip().split()[0].isalpha():
                parts = lines[i].split()
                if len(parts) >= 5:
                    try:
                        angles.append({
                            'id':    int(parts[0]),
                            'type':  int(parts[1]),
                            'atom1': int(parts[2]),
                            'atom2': int(parts[3]),
                            'atom3': int(parts[4])
                        })
                    except ValueError:
                        pass
                i += 1
            continue

        i += 1

    print(f"Parsed: {len(atoms)} atoms, {len(bonds)} bonds, {len(angles)} angles")
    return atoms, bonds, angles, box_bounds, masses, header_info

In [5]:
def write_lammps_data(filename, atoms, bonds, angles, box, masses, header_info):
    """
    Write a LAMMPS data file. Angles section is written only if angles is non-empty.
    Atom IDs are remapped to be contiguous starting from 1.
    """
    has_angles   = len(angles) > 0
    angle_types  = header_info.get('angle_types', 0)
    n_atom_types = max(WALL_TYPE, header_info.get('atom_types', WALL_TYPE))

    # Remap old atom IDs -> new sequential IDs
    old2new = {a['id']: i + 1 for i, a in enumerate(atoms)}

    valid_bonds = [
        {'id': i + 1, 'type': b['type'],
         'atom1': old2new[b['atom1']], 'atom2': old2new[b['atom2']]}
        for i, b in enumerate(bonds)
        if b['atom1'] in old2new and b['atom2'] in old2new
    ]

    valid_angles = []
    if has_angles:
        valid_angles = [
            {'id': i + 1, 'type': a['type'],
             'atom1': old2new[a['atom1']],
             'atom2': old2new[a['atom2']],
             'atom3': old2new[a['atom3']]}
            for i, a in enumerate(angles)
            if a['atom1'] in old2new and a['atom2'] in old2new and a['atom3'] in old2new
        ]
        dropped = len(angles) - len(valid_angles)
        if dropped:
            print(f"  Dropped {dropped} angles referencing removed atoms")

    with open(filename, 'w') as f:
        f.write("LAMMPS data file with lateral walls\n\n")
        f.write(f"{len(atoms)} atoms\n")
        f.write(f"{len(valid_bonds)} bonds\n")
        if has_angles:
            f.write(f"{len(valid_angles)} angles\n")
        f.write("0 dihedrals\n0 impropers\n\n")

        f.write(f"{n_atom_types} atom types\n")
        f.write(f"{header_info.get('bond_types', 1)} bond types\n")
        if has_angles:
            f.write(f"{angle_types} angle types\n")
        f.write("\n")

        f.write(f"{box['xlo']} {box['xhi']} xlo xhi\n")
        f.write(f"{box['ylo']} {box['yhi']} ylo yhi\n")
        f.write(f"{box['zlo']} {box['zhi']} zlo zhi\n\n")

        f.write("Masses\n\n")
        for t in range(1, n_atom_types + 1):
            f.write(f"{t} {masses.get(t, 1.0)}\n")

        f.write("\nAtoms\n\n")
        for i, a in enumerate(atoms, 1):
            f.write(f"{i} {a['mol']} {a['type']} "
                    f"{a['x']:.6f} {a['y']:.6f} {a['z']:.6f}\n")

        if valid_bonds:
            f.write("\nBonds\n\n")
            for b in valid_bonds:
                f.write(f"{b['id']} {b['type']} {b['atom1']} {b['atom2']}\n")

        if has_angles and valid_angles:
            f.write("\nAngles\n\n")
            for a in valid_angles:
                f.write(f"{a['id']} {a['type']} {a['atom1']} {a['atom2']} {a['atom3']}\n")

    print(f"Wrote {filename}")
    print(f"  {len(atoms)} atoms | {len(valid_bonds)} bonds | "
          f"{len(valid_angles) if has_angles else 'N/A (no angles)'} angles")

## Geometry Utilities

In [6]:
def find_min_bounding_rect_angle(hull_points):
    """Rotating calipers: find angle giving minimum-area bounding rectangle."""
    n = len(hull_points)
    min_area  = float('inf')
    best_angle = 0

    for i in range(n):
        edge  = hull_points[(i + 1) % n] - hull_points[i]
        angle = np.arctan2(edge[1], edge[0])
        c, s  = np.cos(-angle), np.sin(-angle)

        rotated = np.column_stack([
            c * hull_points[:, 0] - s * hull_points[:, 1],
            s * hull_points[:, 0] + c * hull_points[:, 1]
        ])
        area = (rotated[:, 0].max() - rotated[:, 0].min()) * \
               (rotated[:, 1].max() - rotated[:, 1].min())

        if area < min_area:
            min_area   = area
            best_angle = angle

    return best_angle


def rotate_mobile_atoms(atoms):
    """Rotate mobile atoms (types 1,2,3) in x-y to align gel with axes."""
    poly = [a for a in atoms if a['type'] in POLYMER_TYPES]
    if not poly:
        raise ValueError("No polymer atoms found")

    xy  = np.array([[a['x'], a['y']] for a in poly])
    com = xy.mean(axis=0)

    hull  = ConvexHull(xy)
    angle = find_min_bounding_rect_angle(xy[hull.vertices])
    print(f"Rotation angle (MABR): {np.degrees(angle):.2f}°")

    c, s = np.cos(-angle), np.sin(-angle)
    for a in atoms:
        if a['type'] in ROTATE_TYPES:
            x = a['x'] - com[0];  y = a['y'] - com[1]
            a['x'] = c*x - s*y + com[0]
            a['y'] = s*x + c*y + com[1]
    return atoms


def find_gel_extent(atoms, clearance, percentile=0.1):
    """Determine lateral gel extent from polymer beads (percentile to exclude outliers)."""
    poly = [a for a in atoms if a['type'] in POLYMER_TYPES]
    xs = np.array([a['x'] for a in poly])
    ys = np.array([a['y'] for a in poly])
    zs = np.array([a['z'] for a in poly])

    xmin = np.percentile(xs, percentile);   xmax = np.percentile(xs, 100 - percentile)
    ymin = np.percentile(ys, percentile);   ymax = np.percentile(ys, 100 - percentile)

    print(f"Polymer x: {xmin:.2f} – {xmax:.2f}  (width {xmax-xmin:.2f})")
    print(f"Polymer y: {ymin:.2f} – {ymax:.2f}  (width {ymax-ymin:.2f})")

    return {
        'xmin': xmin - clearance, 'xmax': xmax + clearance,
        'ymin': ymin - clearance, 'ymax': ymax + clearance,
        'zmin': zs.min(),         'zmax': zs.max()
    }

In [7]:
def remove_external_solvent(atoms, ext):
    """Remove solvent outside lateral wall bounds."""
    kept    = [a for a in atoms if not (
                   a['type'] in SOLVENT_TYPES and
                   (a['x'] < ext['xmin'] or a['x'] > ext['xmax'] or
                    a['y'] < ext['ymin'] or a['y'] > ext['ymax']))]
    removed = len(atoms) - len(kept)
    print(f"Removed {removed} solvent atoms outside walls")
    return kept


def remove_external_polymer(atoms, ext):
    """Remove polymer atoms outside wall boundaries."""
    kept    = [a for a in atoms if not (
                   a['type'] in POLYMER_TYPES and
                   (a['x'] < ext['xmin'] or a['x'] > ext['xmax'] or
                    a['y'] < ext['ymin'] or a['y'] > ext['ymax']))]
    removed = len(atoms) - len(kept)
    print(f"Removed {removed} polymer atoms outside walls")
    return kept


def remove_overlapping_solvent(atoms, wall_atoms, cutoff=1.5):
    """
    Remove solvent atoms that overlap with wall/support/piston atoms.
    Uses a KDTree for efficient spatial queries over large atom counts.
    
    Parameters:
    - atoms: existing atom list (may contain support/piston already)
    - wall_atoms: list of new wall atom dicts (not yet appended to atoms)
    - cutoff: minimum allowed distance between solvent and any rigid atom (σ)
    """
    # Collect ALL rigid-body positions: existing support + piston + new walls
    rigid_positions = []
    for a in atoms:
        if a['type'] in FROZEN_TYPES:
            rigid_positions.append([a['x'], a['y'], a['z']])
    for w in wall_atoms:
        rigid_positions.append([w['x'], w['y'], w['z']])
    
    if not rigid_positions:
        print("No rigid atoms found — skipping overlap removal")
        return atoms
    
    rigid_positions = np.array(rigid_positions)
    tree = KDTree(rigid_positions)
    print(f"Built KDTree with {len(rigid_positions)} rigid-body positions "
          f"(support + piston + walls)")
    
    # Query each solvent atom against the tree
    kept = []
    removed = 0
    for a in atoms:
        if a['type'] in SOLVENT_TYPES:
            pos = np.array([a['x'], a['y'], a['z']])
            dist, _ = tree.query(pos)
            if dist < cutoff:
                removed += 1
                continue
        kept.append(a)
    
    print(f"Removed {removed} solvent atoms within {cutoff}σ of rigid atoms")
    return kept


def generate_wall_atoms(ext, box, spacing=0.2):
    """
    Generate four vertical walls (type 6) as triangular-packed single layers.
    Walls are at xmin, xmax, ymin, ymax and span the full z extent of the box.
    """
    walls       = []
    zlo, zhi    = box['zlo'], box['zhi']
    row_spacing = spacing * np.sqrt(3) / 2

    # ±x walls: triangular lattice in y-z plane
    for x in [ext['xmin'], ext['xmax']]:
        nz = int(np.ceil((zhi - zlo) / row_spacing)) + 1
        for iz in range(nz):
            z        = zlo + iz * row_spacing
            if z > zhi: continue
            y_offset = (spacing / 2) if (iz % 2) else 0
            y = ext['ymin'] + y_offset
            while y <= ext['ymax']:
                walls.append({'type': WALL_TYPE, 'x': x, 'y': y, 'z': z})
                y += spacing

    # ±y walls: triangular lattice in x-z plane
    for y in [ext['ymin'], ext['ymax']]:
        nz = int(np.ceil((zhi - zlo) / row_spacing)) + 1
        for iz in range(nz):
            z        = zlo + iz * row_spacing
            if z > zhi: continue
            x_offset = (spacing / 2) if (iz % 2) else 0
            x = ext['xmin'] + x_offset
            while x <= ext['xmax']:
                walls.append({'type': WALL_TYPE, 'x': x, 'y': y, 'z': z})
                x += spacing

    print(f"Generated {len(walls)} wall atoms (spacing={spacing})")
    return walls

## Run

In [8]:
def add_walls_to_slab(input_file, output_file, wall_spacing, overlap_cutoff=OVERLAP_CUTOFF):
    # --- Parse ---
    atoms, bonds, angles, box, masses, header_info = parse_lammps_data(input_file)
    has_angles = len(angles) > 0
    print(f"Angles in input file: {'yes' if has_angles else 'no'}")

    # --- Rotate mobile atoms to align slab with axes ---
    atoms = rotate_mobile_atoms(atoms)

    # --- Determine wall placement ---
    ext = find_gel_extent(atoms, WALL_CLEARANCE, percentile=0.1)

    # --- Remove atoms outside walls ---
    atoms = remove_external_polymer(atoms, ext)
    atoms = remove_external_solvent(atoms, ext)

    # Note: angles referencing removed atoms are silently dropped during write.
    # For a well-equilibrated slab these should be zero or negligible.

    # --- Generate walls ---
    walls = generate_wall_atoms(ext, box, wall_spacing)

    # --- Remove solvent overlapping with walls/support/piston ---
    atoms = remove_overlapping_solvent(atoms, walls, cutoff=overlap_cutoff)

    # --- Add wall atoms to the atom list ---
    max_id  = max(a['id'] for a in atoms)
    max_mol = max(a['mol'] for a in atoms)
    for i, w in enumerate(walls):
        atoms.append({
            'id':   max_id  + i + 1,
            'mol':  max_mol + i + 1,
            'type': WALL_TYPE,
            'x': w['x'], 'y': w['y'], 'z': w['z']
        })

    # Ensure mass entry for wall type exists
    masses.setdefault(WALL_TYPE, 1.0)

    # --- Write ---
    write_lammps_data(output_file, atoms, bonds, angles, box, masses, header_info)



add_walls_to_slab(input_file, output_file, wall_spacing)

Parsed: 722171 atoms, 115200 bonds, 169196 angles
Angles in input file: yes
Rotation angle (MABR): -178.64°
Polymer x: 5.74 – 57.92  (width 52.18)
Polymer y: 7.88 – 60.07  (width 52.19)
Removed 277 polymer atoms outside walls
Removed 300328 solvent atoms outside walls
Generated 804780 wall atoms (spacing=0.2)
Built KDTree with 931222 rigid-body positions (support + piston + walls)
Removed 20885 solvent atoms within 0.8σ of rigid atoms
  Dropped 567 angles referencing removed atoms
Wrote ../../lammps_data_files_local/walled_slab_support_tall_angle_8_1.0_1.0_10000000.data
  1205461 atoms | 114810 bonds | 168629 angles
